In [2]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import create_urllib3_context
import json
import csv

In [6]:

# 1. 建立一個自訂的 SSL 上下文管理器，允許不安全的重新協商
class LegacyVersionAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        context = create_urllib3_context()
        # 💡 核心：將 SSL 的 Options 加上允許不安全重新協商的旗標
        # 0x40000 代表 SSL_OP_LEGACY_SERVER_CONNECT
        context.options |= 0x40000  
        kwargs["ssl_context"] = context
        return super().init_poolmanager(*args, **kwargs)

session = requests.Session()
session.mount("https://", LegacyVersionAdapter())

def fetch_nccu_courses(url): 
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Origin": "https://qrysub.nccu.edu.tw",
        "Referer": "https://qrysub.nccu.edu.tw/"
    }
    
    try:
        response = session.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error: {e}")
        return None

course_id = set()
def get_course_list(semester):    
    parameters = ["01", "02", "03", "100", "200", "300", "400", "500", "600", "700", "800", "900", "Z23", "ZA0", "ZC0"]
    data = []
    for parameter in parameters: 
        response = fetch_nccu_courses(f"https://es.nccu.edu.tw/course/zh-TW/:sem={semester}%20:dp1={parameter}%20/")
        if response:
            for c in response:
                if c["subNum"] not in course_id:
                    data.append(c)
                    course_id.add(c["subNum"])

    return data

data = []
for semester in ["1142", "1141", "1132", "1131", "1122", "1121", "1112", "1111", "1102", "1101"]:
    data += get_course_list(semester)
print(len(data))

10232


In [20]:
with open("response.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [8]:
gdeType = set()
lmtKind = set()
tranTpe = set()
subGde  = set()
subKind = set()

for course in data:
    if ("學士" in course["gdeType"]):
        gdeType.add(course["gdeType"])
        lmtKind.add(course["lmtKind"])
        subGde.add(course["subGde"])
        subKind.add(course["subKind"])

print(f"gdeType: {gdeType}")
print(f"lmtKind: {lmtKind}")
print(f"subGde: {subGde}")
print(f"subKind: {subKind}")

gdeType: {'學士班、碩士班', '學士班'}
lmtKind: {'', '外文通識', '社會通識', '跨領域(社會、資訊)', '跨領域(人文、社會、資訊)', '跨領域(社會、自然)', '中文通識', '跨領域(人文、社會)', '書院通識', '跨領域(人文、資訊)', '跨領域(人文、社會、自然)', '跨領域(社會、自然、資訊)', '跨領域(人文、自然)', '跨領域(自然、資訊)', '自然通識', '人文通識', '資訊通識'}
subGde: {'社科院', '資科四資碩工一資碩計一', '傳院二甲傳院二乙傳院二丙傳院二丁', '土文三土文四', '文院碩士文院學士', '斯語一甲斯語一乙', '斯語四甲斯語四乙', '韓文一韓文二韓文三韓文四', '教院碩士教院學士', '東南越三東南泰三東南印三東南越四東南泰四東南印四', '會計三甲會計三乙', '企管一乙', '廣電四', '廣告系', '生科程碩生科程學', '日文四', '電物學程', '教育一', '東南學四', '中文三甲中文三乙中文四甲中文四乙', '風管系', '東南泰二東南印二東南泰三東南印三東南泰四東南印四', '韓文二', '企管一甲', '中文系', '創國學一創國學二創國學三創國學四', '心理二', '會計一甲', '傳院一丁', '創國學士', '應數四', '資管二甲資管二乙', '會計三甲', '廣告三', '國貿四甲', '統計二', '歐文法一歐文德一歐文西一歐文法二歐文德二歐文西二歐文法三歐文德三歐文西三歐文法四歐文德四歐文西四', '華語文中心', '斯語一甲斯語一乙斯語二甲斯語二乙', '政治一政治二', '歐文法一', '企管三甲企管三乙企管四甲企管四乙', '中文三乙', '東南泰三東南泰四', '中文一甲', '歐文法四', '土文系', '國務學士', '國貿二甲國貿二乙', '統計一', '歐文德二', '經濟一甲經濟一乙經濟二甲經濟二乙經濟三甲經濟三乙', '新聞四', '歐文西二', '地四土管', '東南越二', '傳院一甲傳院一乙傳院一丙傳院一丁', '財管四', '應數一應數二', '社工所', '台文所', '法學院', '法律系', '遠距通', '東南越三東南學四', '民族二民族三', '哲學一哲學二哲學三'

In [ ]:
department_keywords = {
    '文學院': '100', '中文': '101', '教育': '102', '歷史': '103', '哲學': '104',
    '政治': '202', '外交': '203', '社會': '204', '財政': '205', '公行': '206', '地': '207', '經濟': '208', '民族': '209', 
    '國貿': '301', '金融': '302', '會計': '303', '統計': '304', '企管': '305', '資管': '306', '財管': '307', '風管': '308',
    '新聞': '401', '廣告': '402', '廣電': '403', '傳院': '405', '傳播': '405',
    '英文': '501', '阿文': '502', '斯語': '504', '日文': '506', '韓文': '507', '土文': '508', '歐文': '509', '東南泰': '510',
    '法律': '601',
    '應數': '701', '心理': '702', '資訊': '703',
    '創國': 'ZU1',
    '體育': 'B00', '華語文': "162", "外文中": "5T1"
}

course_type_keywords = {
    '群修': 'P', '必修': 'R', '選修': 'E', 
    "中文通識": "GC", "外文通識": "GF", 
    "人文通識": "GH", "社會通識": "GS", "自然通識": "GN", "資訊通識": "GI", "書院通識": "GA",
    "跨領域(社會、自然)": "GSN", "跨領域(人文、社會)": "GHS", "跨領域(人文、自然)": "GHN", "跨領域(人文、資訊)": "GHI", "跨領域(社會、資訊)": "GSI", "跨領域(自然、資訊)": "GNI",
    "跨領域(社會、自然、資訊)": "GSNI", "跨領域(人文、社會、自然)": "GHSN", "跨領域(人文、社會、資訊)": "GHSI",
}

course_info = [['course_id', 'course_name', 'course_type', 'teacher_name', 'department_id', 'credits']]
for course in data:
    if ("學士" in course["gdeType"]):
        course_id = course["subNum"]
        course_name = course["subNam"]
        if "體育" in course_name:
            course_type = course_type_keywords[course["subKind"]] + "PE"        
        elif "國防" in course_name:
            course_type = "CD"  
        elif course["subKind"]:
            course_type = course_type_keywords[course["subKind"]]
        else:
            course_type = course_type_keywords[course["lmtKind"]]
        if course["core"] == "是":
            course_type = "C" + course_type
            
        teacher_name = course["teaNam"]
        
        for key, val in department_keywords.items():
            if key in course["subGde"]:
                department_id = str(val)
                break
        else:
            department_id = "000"
        credits = int(float(course["subPoint"]))      
        
        course_info.append([
            course_id,
            course_name,
            course_type,
            teacher_name,
            department_id,
            credits
        ])

with open('course_information.csv', 'w', encoding='utf-8', newline="") as f:
    writer = csv.writer(f)
    writer.writerows(course_info)